# 부모 문서 검색(Parent-document retrieval)

작은 자식 청크를 임베딩해 검색 정밀도를 높이되, 최종 결과에는 더 큰 부모
문서를 반환해 답변 문맥을 보존합니다. `ParentDocumentRetriever`는 v1에서
classic 패키지로 이동했으므로, 같은 알고리즘을 Chroma와 명시적인 부모 저장소로
구현합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain-core==1.6.3" "langchain-openai==1.6.2" \
#   "langchain-chroma==1.1.0" "langchain-text-splitters==1.1.2" \
#   python-dotenv


In [ ]:
import getpass
import os
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

data_path = Path("data/appendix-keywords.txt")
if not data_path.exists():
    raise FileNotFoundError(f"실습 파일이 없습니다: {data_path.resolve()}")

source_docs = [
    Document(
        page_content=data_path.read_text(encoding="utf-8"),
        metadata={"source": str(data_path)},
    )
]
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


## 재사용 가능한 부모-자식 인덱서

`parent_splitter=None`이면 원문 전체가 부모입니다. 부모 분할기를 전달하면 먼저
큰 청크를 만들고, **각 부모 안에서** 자식 청크를 만듭니다. 이 매핑 순서가
부모 문맥을 정확히 복원하는 핵심입니다.


In [ ]:
def build_parent_index(
    documents: list[Document],
    *,
    child_splitter: RecursiveCharacterTextSplitter,
    parent_splitter: RecursiveCharacterTextSplitter | None = None,
    child_k: int = 4,
) -> tuple[Chroma, dict[str, Document], RunnableLambda]:
    parents = (
        parent_splitter.split_documents(documents)
        if parent_splitter
        else list(documents)
    )
    parent_store: dict[str, Document] = {}
    children: list[Document] = []
    child_ids: list[str] = []

    for parent_index, parent in enumerate(parents):
        parent_id = uuid4().hex
        stored_parent = Document(
            page_content=parent.page_content,
            metadata={
                **parent.metadata,
                "parent_id": parent_id,
                "parent_index": parent_index,
            },
        )
        parent_store[parent_id] = stored_parent

        for child_index, child in enumerate(
            child_splitter.split_documents([stored_parent])
        ):
            children.append(
                Document(
                    page_content=child.page_content,
                    metadata={
                        **child.metadata,
                        "parent_id": parent_id,
                        "child_index": child_index,
                    },
                )
            )
            child_ids.append(uuid4().hex)

    vectorstore = Chroma.from_documents(
        children,
        embeddings,
        collection_name=f"parent-child-{uuid4().hex}",
        ids=child_ids,
        collection_configuration=CHROMA_CONFIGURATION,
    )

    def retrieve_parents(query: str) -> list[Document]:
        child_hits = vectorstore.similarity_search(query, k=child_k)
        ordered_parent_ids: list[str] = []
        for hit in child_hits:
            parent_id = hit.metadata["parent_id"]
            if parent_id not in ordered_parent_ids:
                ordered_parent_ids.append(parent_id)
        return [parent_store[parent_id] for parent_id in ordered_parent_ids]

    retriever = RunnableLambda(retrieve_parents).with_config(
        {"run_name": "parent_document_retriever"}
    )
    return vectorstore, parent_store, retriever


## 1. 원문 전체를 부모로 반환


In [ ]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    add_start_index=True,
)
full_vectorstore, full_parent_store, full_parent_retriever = build_parent_index(
    source_docs,
    child_splitter=child_splitter,
)

child_hits = full_vectorstore.similarity_search("Word2Vec", k=2)
parent_hits = full_parent_retriever.invoke("Word2Vec")

print("자식 청크 길이:", [len(doc.page_content) for doc in child_hits])
print("반환된 부모 길이:", [len(doc.page_content) for doc in parent_hits])
print(parent_hits[0].page_content[:500])


## 2. 큰 청크를 부모로 반환

원문 전체가 너무 길다면 1,000자 부모 → 200자 자식 계층을 만듭니다. 검색은
자식에서 수행하지만 답변 체인에는 관련 부모 청크만 전달합니다.


In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1_000,
    chunk_overlap=100,
    add_start_index=True,
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    add_start_index=True,
)
chunk_vectorstore, chunk_parent_store, chunk_parent_retriever = build_parent_index(
    source_docs,
    parent_splitter=parent_splitter,
    child_splitter=child_splitter,
    child_k=6,
)

print(f"부모 수: {len(chunk_parent_store)}")
print(
    "자식 벡터 검색 결과:",
    chunk_vectorstore.similarity_search("Word2Vec", k=1)[0].page_content,
)
print("\n부모 검색 결과:")
for doc in chunk_parent_retriever.invoke("Word2Vec"):
    print(doc.page_content[:500], "\n", "-" * 80)
